<a href="https://colab.research.google.com/github/rahilkhan-acadmic/APAIML-GradedMiniProject/blob/develop/capstone/build_features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 2 — Exploratory Data Analysis and Data Quality
1. Load data into Python for analysis. Combine them
2. Analyze missing values in approved_amount, payment_days, and length_of_stay_hours.
3. Perform distribution analysis by department, visit_type, insurance_provider, and city.
4. Detect and classify outliers in billed_amount, payment_days, and length_of_stay_hours.
4. Engineer features such as visit frequency, average length of stay per patient, provider rejection rate, days since registration, and time-based features.


In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Upload file to google.colab
import io
from google.colab import files, drive

drive.mount('/content/drive')

# comment this code, if files already uploaded.
# uploaded = files.upload()

Mounted at /content/drive


### 1. Load data into Python for analysis. Combine them


In [3]:
# Load and read patients, visits & billing csv
patients_df = pd.read_csv('/content/drive/MyDrive/files/capstone/patients.csv')
patients_df.head()

visits_df = pd.read_csv('/content/drive/MyDrive/files/capstone/visits.csv')
visits_df.head()

billing_df = pd.read_csv('/content/drive/MyDrive/files/capstone/billing.csv')
billing_df.head()

,bill_id,visit_id,billed_amount,approved_amount,claim_status,payment_days,billing_date
0,1,1,23577.37,0.00,Rejected,16.0,2025-06-18
1,2,2,38178.81,38178.81,Paid,18.0,2025-10-09
2,3,3,5038.97,5038.97,Paid,NaN,2025-01-20
3,4,4,22813.34,22813.34,Paid,16.0,2025-12-24
4,5,5,27106.95,27106.95,Paid,14.0,2025-09-23


### 2. Analyze missing values in approved_amount, payment_days, and length_of_stay_hours.


In [4]:
# Analyze missing values in approved_amount, payment_days, and length_of_stay_hours.
df_merged = pd.merge(patients_df, visits_df, on='patient_id', how = 'inner')
df_merged = pd.merge(df_merged, billing_df, on='visit_id', how = 'inner')
df_merged.head()


,patient_id,age,gender,city,insurance_provider,chronic_flag,registration_date,visit_id,visit_date,department,visit_type,length_of_stay_hours,risk_score,doctor_id,bill_id,billed_amount,approved_amount,claim_status,payment_days,billing_date
0,1,53,M,Hyderabad,SecureLife,0,2025-05-14,4327,2025-10-15,General,ICU,6.39,High,162,4327,26322.05,13938.52,Pending,NaN,2025-11-29
1,1,53,M,Hyderabad,SecureLife,0,2025-05-14,5395,2025-09-21,Orthopedics,OPD,39.12,Low,181,5395,13258.56,0.00,Rejected,3.0,2025-08-15
2,1,53,M,Hyderabad,SecureLife,0,2025-05-14,9032,2025-02-09,Cardiology,OPD,19.57,Medium,121,9032,26555.09,26555.09,Paid,4.0,2025-12-31
3,1,53,M,Hyderabad,SecureLife,0,2025-05-14,9506,2025-10-09,Neurology,ER,30.32,Medium,135,9506,11583.16,8065.34,Pending,6.0,2025-01-30
4,1,53,M,Hyderabad,SecureLife,0,2025-05-14,10868,2025-07-07,ICU,ER,30.07,High,164,10868,9049.72,9049.72,Paid,19.0,2025-02-26


###5. Engineer features such as visit frequency, average length of stay per patient, provider rejection rate, days since registration, and time-based features.


In [5]:
# Ensure date columns are in datetime format
df_merged['registration_date'] = pd.to_datetime(df_merged['registration_date'])
df_merged['visit_date'] = pd.to_datetime(df_merged['visit_date'])
df_merged['billing_date'] = pd.to_datetime(df_merged['billing_date'])

# 1. Visit Frequency per Patient
visit_frequency = df_merged.groupby('patient_id')['visit_id'].nunique().reset_index()
visit_frequency.rename(columns={'visit_id': 'visit_frequency'}, inplace=True)
df_merged = pd.merge(df_merged, visit_frequency, on='patient_id', how='left')
print("Visit Frequency per Patient (first 5 rows):")
display(df_merged[['patient_id', 'visit_frequency']].head())

# 2. Average Length of Stay per Patient
avg_los_per_patient = df_merged.groupby('patient_id')['length_of_stay_hours'].mean().reset_index()
avg_los_per_patient.rename(columns={'length_of_stay_hours': 'avg_length_of_stay_hours'}, inplace=True)
df_merged = pd.merge(df_merged, avg_los_per_patient, on='patient_id', how='left')
print("\nAverage Length of Stay per Patient (first 5 rows):")
display(df_merged[['patient_id', 'avg_length_of_stay_hours']].head())

# 3. Provider Rejection Rate
total_claims_per_provider = df_merged.groupby('insurance_provider')['bill_id'].count().reset_index()
total_claims_per_provider.rename(columns={'bill_id': 'total_claims'}, inplace=True)

rejected_claims_per_provider = df_merged[df_merged['claim_status'] == 'Rejected'].groupby('insurance_provider')['bill_id'].count().reset_index()
rejected_claims_per_provider.rename(columns={'bill_id': 'rejected_claims'}, inplace=True)

provider_rejection = pd.merge(total_claims_per_provider, rejected_claims_per_provider, on='insurance_provider', how='left')
provider_rejection['rejected_claims'] = provider_rejection['rejected_claims'].fillna(0)
provider_rejection['rejection_rate'] = (provider_rejection['rejected_claims'] / provider_rejection['total_claims']) * 100
df_merged = pd.merge(df_merged, provider_rejection[['insurance_provider', 'rejection_rate']], on='insurance_provider', how='left')
print("\nProvider Rejection Rate (first 5 rows):")
display(df_merged[['insurance_provider', 'rejection_rate']].head())


# 4. Days Since Registration for each visit
df_merged['days_since_registration'] = (df_merged['visit_date'] - df_merged['registration_date']).dt.days
print("\nDays Since Registration (first 5 rows):")
display(df_merged[['patient_id', 'registration_date', 'visit_date', 'days_since_registration']].head())

# 5. Time-based Features from visit_date
df_merged['visit_year'] = df_merged['visit_date'].dt.year
df_merged['visit_month'] = df_merged['visit_date'].dt.month
df_merged['visit_day'] = df_merged['visit_date'].dt.day
df_merged['visit_day_of_week'] = df_merged['visit_date'].dt.dayofweek # Monday=0, Sunday=6
df_merged['visit_quarter'] = df_merged['visit_date'].dt.quarter
print("\nTime-based Features (first 5 rows):")
display(df_merged[['visit_date', 'visit_year', 'visit_month', 'visit_day', 'visit_day_of_week', 'visit_quarter']].head())
## Billing days since registration
df_merged['billing_days_since_registration'] = (df_merged['billing_date'] - df_merged['registration_date']).dt.days
display(df_merged['billing_days_since_registration'].reset_index())


Visit Frequency per Patient (first 5 rows):


,patient_id,visit_frequency
0,1,6
1,1,6
2,1,6
3,1,6
4,1,6



Average Length of Stay per Patient (first 5 rows):


,patient_id,avg_length_of_stay_hours
0,1,22.183333
1,1,22.183333
2,1,22.183333
3,1,22.183333
4,1,22.183333



Provider Rejection Rate (first 5 rows):


...


Days Since Registration (first 5 rows):


,patient_id,registration_date,visit_date,days_since_registration
0,1,2025-05-14,2025-10-15,154
1,1,2025-05-14,2025-09-21,130
2,1,2025-05-14,2025-02-09,-94
3,1,2025-05-14,2025-10-09,148
4,1,2025-05-14,2025-07-07,54



Time-based Features (first 5 rows):


,visit_date,visit_year,visit_month,visit_day,visit_day_of_week,visit_quarter
0,2025-10-15,2025,10,15,2,4
1,2025-09-21,2025,9,21,6,3
2,2025-02-09,2025,2,9,6,1
3,2025-10-09,2025,10,9,3,4
4,2025-07-07,2025,7,7,0,3


,index,billing_days_since_registration
0,0,199
1,1,93
2,2,231
3,3,-104
4,4,-77
...,...,...
24995,24995,-121
24996,24996,31
24997,24997,16
24998,24998,173
